# 4. Value Iteration
### Finds Optimal State Values

**Value Iteration** combines policy evaluation and policy improvement into a single step by turning the **Bellman Optimality Equation** into an iterative update rule.

#### Mathematical Foundation:
1. **Bellman Optimality Equation**:
   $$V^*(s) = \max_{a} \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V^*(s') \right]$$
2. **Value Iteration Update**:
   $$V_{k+1}(s) \leftarrow \max_{a} \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V_k(s') \right]$$
3. **Key Advantage over Policy Iteration**:
   Does not wait for policy evaluation to converge before performing greedy improvement; it truncates policy evaluation after **exactly 1 sweep**!


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Define a 4x4 GridWorld MDP Environment
# States: 16 grid cells (0 to 15), where (0,0) [state 0] and (3,3) [state 15] are terminal
# Actions: 0: UP, 1: RIGHT, 2: DOWN, 3: LEFT

class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.num_states = size * size
        self.num_actions = 4
        self.actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
        self.terminal_states = {0, self.num_states - 1}
        
    def get_next_state_and_reward(self, state, action):
        if state in self.terminal_states:
            return state, 0.0
            
        r, c = divmod(state, self.size)
        if action == 0:    # UP
            r = max(r - 1, 0)
        elif action == 1:  # RIGHT
            c = min(c + 1, self.size - 1)
        elif action == 2:  # DOWN
            r = min(r + 1, self.size - 1)
        elif action == 3:  # LEFT
            c = max(c - 1, 0)
            
        next_state = r * self.size + c
        reward = -1.0  # -1 step cost until terminal goal reached
        return next_state, reward

env = GridWorld()
print(f"GridWorld initialized with {env.num_states} states and {env.num_actions} actions.")


In [ ]:
# Step 3: Value Iteration Implementation
def value_iteration(env, gamma=1.0, theta=1e-4):
    V = np.zeros(env.num_states)
    iteration = 0
    delta_history = []
    
    while True:
        delta = 0.0
        V_new = np.copy(V)
        
        for s in range(env.num_states):
            if s in env.terminal_states:
                continue
                
            # Compute action values for all actions
            action_values = np.zeros(env.num_actions)
            for a in range(env.num_actions):
                next_s, r = env.get_next_state_and_reward(s, a)
                action_values[a] = r + gamma * V[next_s]
                
            # Bellman Optimality Backup (Max over actions)
            V_new[s] = np.max(action_values)
            delta = max(delta, abs(V[s] - V_new[s]))
            
        V = V_new
        delta_history.append(delta)
        iteration += 1
        
        if delta < theta:
            print(f"Value Iteration converged in {iteration} sweeps (threshold = {theta}).")
            break
            
    # Extract optimal deterministic policy from V*
    policy = np.zeros((env.num_states, env.num_actions))
    for s in range(env.num_states):
        if s in env.terminal_states:
            continue
        action_values = [env.get_next_state_and_reward(s, a)[1] + gamma * V[env.get_next_state_and_reward(s, a)[0]] for a in range(env.num_actions)]
        policy[s, np.argmax(action_values)] = 1.0
        
    return V, policy, delta_history

V_star, pi_star, deltas = value_iteration(env)


In [ ]:
# Step 4: Plot Maximum Value Change (Delta) per Sweep
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(deltas) + 1), deltas, marker='o', color='purple')
plt.xlabel("Iteration Sweep")
plt.ylabel("Delta (Max change in V)")
plt.title("Value Iteration: Convergence of Bellman Optimality Operator")
plt.grid(True)
plt.show()


In [ ]:
# Function to plot State-Value Heatmap and Policy Arrows
def plot_grid(V, policy=None, title="State-Value Grid"):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_v = V.reshape((4, 4))
    
    cax = ax.matshow(grid_v, cmap='Blues_r')
    fig.colorbar(cax)
    
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            val_text = f"{V[s]:.1f}"
            if s in [0, 15]:
                val_text += "\n(Goal)"
            elif policy is not None:
                best_a = np.argmax(policy[s])
                val_text += f"\n{arrows[best_a]}"
                
            ax.text(c, r, val_text, ha='center', va='center', color='black', fontsize=11, weight='bold')
            
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    plt.title(title, pad=20)
    plt.show()


In [ ]:
# Step 6: Visualize Optimal Value Grid and Extracted Policy
plot_grid(V_star, policy=pi_star, title="Value Iteration: Optimal Value Function V* and Policy π*")


### Summary
- Direct application of Bellman optimality backup.
- Replaces inner convergence loops with a single max operation per state sweep.
- Guaranteed geometric convergence rate governed by discount factor $\gamma$.
